In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [9]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_name = "meta-llama/Llama-3.2-1B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype="auto"
)
model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 5421.64it/s]


In [10]:
print(model.get_memory_footprint()/1e6)

2471.629056


In [11]:
GENERATION_CONFIGS = {
    "thinking": {
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.0,
        "max_new_tokens": 2048,
    },
    "non_thinking": {
        "do_sample": True,
        "temperature": 0.1,
        "top_p": 0.8,
        "top_k": 20,
        "min_p": 0.0,
        "max_new_tokens": 1024,
    },
}

In [12]:
def format_prompt(
    question: str,
    task_type: str = "general",
) -> str:
    question = question.strip()

    if task_type == "math":
        return (
            f"{question}\n\n"
            "Please reason step by step, and put your final answer within \\boxed{}."
        )

    if task_type == "multiple_choice":
        return (
            f"{question}\n\n"
            "Please show your choice in the answer field with only the choice letter, "
            'e.g., "answer": "C".'
        )

    if task_type == "general":
        return question

    raise ValueError(f"Unsupported task_type: {task_type}")

In [13]:
question = "Choose an answer for the following question and give your reasons.\n\nQuestion:\nWhich figure of speech is used in this text?\nLuke's room is as tidy as an overgrown garden.\n\nChoices:\nA. verbal irony\nB. pun\n\nAnswer:"
task_type = "multiple_choice"

prompt = format_prompt(
    question=question,
    task_type=task_type
)

messages = [
    {"role": "user", "content": prompt},
]

thinking = True
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=thinking,
)

print(text)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 28 Apr 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Choose an answer for the following question and give your reasons.

Question:
Which figure of speech is used in this text?
Luke's room is as tidy as an overgrown garden.

Choices:
A. verbal irony
B. pun

Answer:

Please show your choice in the answer field with only the choice letter, e.g., "answer": "C".<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [14]:
inputs = tokenizer(
    text,
    return_tensors="pt",
).to(device)

mode = "thinking" if thinking else "non_thinking"

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        **GENERATION_CONFIGS[mode],
        pad_token_id=tokenizer.eos_token_id,
    )

generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True,
)

In [15]:
print(response.strip())

answer: B
